# Workshop: S2S – Accessing Sub-National Data

**Duration:** 30 minutes  
**Type:** Hands-on exercise  
**Presenter:** Gabe

---

### What you’ll learn

1. How to install and set up the Space2Stats Python client
2. How to fetch administrative boundaries for any country
3. How to discover pre-computed sub-national datasets on the World Bank DDH
4. How to query and visualize population, nighttime lights, urbanization, and flood exposure data

### Key takeaway

> *Pre-computed ADM2 summaries on the World Bank Data Distribution Hub (DDH) give you ready-to-use sub-national indicators — no rasters, no zonal stats, no GIS software required.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/worldbank/DECAT_Space2Stats/blob/glevin_workshop_prep/workshops/01_accessing_subnational_data.ipynb)

In [ ]:
# Install the client
%pip install space2stats-client matplotlib mapclassify

In [ ]:
from space2stats_client import Space2StatsClient
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# Initialize the client — that’s it, no auth required
client = Space2StatsClient()

---
## Part 2: Fetch Admin Boundaries (5 min)

The client has a built-in helper to fetch official World Bank administrative boundaries.

In [ ]:
# Fetch ADM2 (district-level) boundaries for Rwanda
ISO3 = "RWA"
boundaries = client.fetch_admin_boundaries(ISO3, "ADM2")
print(f"Number of districts: {len(boundaries)}")
boundaries.plot(figsize=(8, 8), edgecolor="black", linewidth=0.3, color="lightblue")
plt.title(f"{ISO3} – ADM2 Boundaries")
plt.axis("off")
plt.show()

In [ ]:
# What does the boundary data look like?
boundaries.head()

In [ ]:
# You can also fetch ADM1 (province-level) boundaries
adm1 = client.fetch_admin_boundaries(ISO3, "ADM1")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

adm1.plot(ax=axes[0], edgecolor="black", linewidth=0.5, color="lightyellow")
axes[0].set_title(f"{ISO3} – ADM1 ({len(adm1)} provinces)", fontsize=13)
axes[0].axis("off")

boundaries.plot(ax=axes[1], edgecolor="black", linewidth=0.3, color="lightblue")
axes[1].set_title(f"{ISO3} – ADM2 ({len(boundaries)} districts)", fontsize=13)
axes[1].axis("off")

plt.tight_layout()
plt.show()

---
## Part 3: Discover DDH Datasets (5 min)

The [World Bank Development Data Hub (DDH)](https://datacatalog.worldbank.org/) hosts pre-computed ADM2-level summaries derived from Space2Stats. These are ready-to-use — no hexagons, no spatial joins, just district-level indicators.

Let’s see what’s available.

In [ ]:
# What pre-computed ADM2 datasets are available?
client.get_adm2_dataset_info()

Four datasets are currently available:

| Dataset | Description |
|---------|-------------|
| `population` | Gridded population by gender and age group (WorldPop 2020) |
| `nighttimelights` | Satellite-measured luminosity (VIIRS, 2012–2024) |
| `urbanization` | Settlement classification (GHS-SMOD) |
| `flood_exposure` | Population exposed to flood risk (Fathom v3) |

---
## Part 4: Query DDH Summaries (15 min)

### 4a. Population

In [ ]:
pop = client.get_adm2_summaries("population", iso3_filter="RWA")

# DDH API returns strings — convert numeric columns
num_cols = [c for c in pop.columns if c != "ADM2CD_c" and c != "ISO3"]
pop[num_cols] = pop[num_cols].apply(pd.to_numeric)

print(f"Columns: {len(pop.columns)}")
pop.head(3)

In [ ]:
# Join with boundaries and map total population
pop_geo = boundaries.merge(pop, on="ADM2CD_c")

# Round population to clean numbers for display
pop_geo["sum_pop_2020"] = (pop_geo["sum_pop_2020"] / 1000).round() * 1000

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pop_geo.plot(
    column="sum_pop_2020", cmap="YlOrRd", scheme="quantiles", k=5,
    legend=True, edgecolor="black", linewidth=0.3, ax=axes[0],
    legend_kwds={"title": "Total Pop", "loc": "lower left", "fmt": "{:.0f}"}
)
axes[0].set_title("Total Population (2020)", fontsize=13)
axes[0].axis("off")

# Female share
pop_geo["female_share"] = pop_geo["sum_pop_f_2020"] / pop_geo["sum_pop_2020"] * 100

pop_geo.plot(
    column="female_share", cmap="PuOr", legend=True,
    edgecolor="black", linewidth=0.3, ax=axes[1],
    legend_kwds={"label": "Female %"}
)
axes[1].set_title("Female Population Share (%)", fontsize=13)
axes[1].axis("off")

plt.suptitle("Rwanda — Population (DDH)", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 4b. Nighttime Lights

The NTL dataset has yearly columns from 2012 to 2024, making it easy to visualize trends over time.

In [ ]:
ntl = client.get_adm2_summaries("nighttimelights", iso3_filter="RWA")

# DDH API returns strings — convert numeric columns
num_cols = [c for c in ntl.columns if c != "ADM2CD_c" and c != "ISO3"]
ntl[num_cols] = ntl[num_cols].apply(pd.to_numeric)

ntl.head(3)

In [ ]:
# Join with boundaries for district names
ntl_geo = boundaries[["ADM2CD_c", "NAM_2"]].merge(ntl, on="ADM2CD_c")

# Reshape wide-to-long for plotting (up to 2023)
ntl_cols = [c for c in ntl.columns if c.startswith("sum_viirs_ntl_") and int(c[-4:]) <= 2023]
ntl_long = ntl_geo.melt(
    id_vars=["ADM2CD_c", "NAM_2"],
    value_vars=ntl_cols,
    var_name="year_col",
    value_name="ntl"
)
ntl_long["year"] = ntl_long["year_col"].str.extract(r"(\d{4})").astype(int)

# Pick a few districts to compare
highlight = ["Kicukiro", "Gasabo", "Nyarugenge", "Musanze", "Huye"]
ntl_subset = ntl_long[ntl_long["NAM_2"].isin(highlight)]

fig, ax = plt.subplots(figsize=(12, 5))
for name, group in ntl_subset.groupby("NAM_2"):
    ax.plot(group["year"], group["ntl"], marker="o", label=name)

ax.set_xlabel("Year")
ax.set_ylabel("Total Nighttime Lights (sum VIIRS)")
ax.set_title("Nighttime Lights Trends by District (2012–2023)")
ax.legend(title="District")
plt.tight_layout()
plt.show()

### 4c. Urbanization

The urbanization dataset uses the GHS-SMOD settlement classification. Each hexagon is classified into settlement types, and both **area counts** and **population** are provided per class.

| Code | Settlement type |
|------|----------------|
| 11 | Very low density rural |
| 12 | Low density rural |
| 13 | Rural cluster |
| 21 | Suburban |
| 22 | Semi-dense urban |
| 23 | Dense urban |
| 30 | Urban centre |

In [ ]:
urb = client.get_adm2_summaries("urbanization", iso3_filter="RWA")

# DDH API returns strings — convert numeric columns
num_cols = [c for c in urb.columns if c != "ADM2CD_c" and c != "ISO3"]
urb[num_cols] = urb[num_cols].apply(pd.to_numeric)

urb.head(3)

In [ ]:
# Join with boundaries
urb_geo = boundaries[["ADM2CD_c", "NAM_2", "geometry"]].merge(urb, on="ADM2CD_c")
urb_geo = gpd.GeoDataFrame(urb_geo, geometry="geometry")

# Calculate urban population share (classes 22-30 are urban/suburban)
urb_geo["urban_pop"] = (
    urb_geo["ghs_22_pop"] +
    urb_geo["ghs_23_pop"] + urb_geo["ghs_30_pop"]
)
urb_geo["urban_share"] = urb_geo["urban_pop"] / urb_geo["ghs_total_pop"] * 100

fig, ax = plt.subplots(figsize=(8, 8))
urb_geo.plot(
    column="urban_share", cmap="YlGn", scheme="quantiles", k=5,
    legend=True, edgecolor="black", linewidth=0.3, ax=ax,
    legend_kwds={"title": "Urban %", "loc": "lower left"}
)
ax.set_title("Urban Population Share by District (%)", fontsize=14)
ax.axis("off")
plt.show()

### 4d. Flood Exposure

The flood exposure dataset shows population at risk from flooding (depth > 15 cm in a 1-in-100-year event).

In [ ]:
flood = client.get_adm2_summaries("flood_exposure", iso3_filter="RWA")

# DDH API returns strings — convert numeric columns
num_cols = [c for c in flood.columns if c != "ADM2CD_c" and c != "ISO3"]
flood[num_cols] = flood[num_cols].apply(pd.to_numeric)

flood.head(3)

In [ ]:
# Join with boundaries
flood_geo = boundaries[["ADM2CD_c", "NAM_2", "geometry"]].merge(flood, on="ADM2CD_c")
flood_geo = gpd.GeoDataFrame(flood_geo, geometry="geometry")

# Round to clean numbers
flood_geo["pop_flood"] = (flood_geo["pop_flood"] / 1000).round() * 1000

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

flood_geo.plot(
    column="pop_flood", cmap="Blues", scheme="quantiles", k=5,
    legend=True, edgecolor="black", linewidth=0.3, ax=axes[0],
    legend_kwds={"title": "Pop exposed", "loc": "lower left", "fmt": "{:.0f}"}
)
axes[0].set_title("Population Exposed to Floods", fontsize=13)
axes[0].axis("off")

flood_geo.plot(
    column="pop_flood_pct", cmap="Reds", scheme="quantiles", k=5,
    legend=True, edgecolor="black", linewidth=0.3, ax=axes[1],
    legend_kwds={"title": "% exposed", "loc": "lower left", "fmt": "{:.1f}"}
)
axes[1].set_title("Flood Exposure Rate (%)", fontsize=13)
axes[1].axis("off")

plt.suptitle("Rwanda — Flood Exposure (DDH)", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---
## Part 5: Your Turn! (5 min)

### Challenge

Pick a **different country** and explore the DDH datasets. Some ideas:

| Country | ISO3 | Try... |
|---------|------|--------|
| Colombia | COL | Compare NTL trends across departments |
| Bangladesh | BGD | Map flood exposure — which districts are most at risk? |
| Nigeria | NGA | Urbanization patterns — where is growth concentrated? |
| Sri Lanka | LKA | Population pyramid by district |

**Steps:**
1. Fetch boundaries: `client.fetch_admin_boundaries("YOUR_ISO3", "ADM2")`
2. Pick a DDH dataset: `client.get_adm2_summaries("dataset", iso3_filter="YOUR_ISO3")`
3. Join with boundaries and visualize!

In [ ]:
# Your code here!


---
## Recap

| What we did | Method |
|---|---|
| Fetched admin boundaries | `fetch_admin_boundaries(iso3, adm)` |
| Listed available DDH datasets | `get_adm2_dataset_info()` |
| Queried population by district | `get_adm2_summaries("population", iso3_filter)` |
| Tracked nighttime lights trends | `get_adm2_summaries("nighttimelights", iso3_filter)` |
| Mapped urbanization patterns | `get_adm2_summaries("urbanization", iso3_filter)` |
| Assessed flood exposure | `get_adm2_summaries("flood_exposure", iso3_filter)` |